##Install Requirement

In [ ]:
!nvidia-smi

In [ ]:
!git pull origin main

In [ ]:
!git clone https://github.com/alexandrachirita98/MedViT-Quantum/

In [2]:
%cd /kaggle/working/MedViT-Quantum

/kaggle/working/MedViT-Quantum


In [ ]:
%pwd

In [ ]:
pip install -r requirements.txt

In [3]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

import torchvision
import torchvision.utils
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torchsummary import summary

from tqdm import tqdm
import medmnist
from medmnist import INFO, Evaluator

import torchattacks
from torchattacks import PGD, FGSM

In [5]:
print("PyTorch", torch.__version__)
print("Torchvision", torchvision.__version__)
print("Torchattacks", torchattacks.__version__)
print("Numpy", np.__version__)
print("Medmnist", medmnist.__version__)

PyTorch 2.10.0+cu128
Torchvision 0.25.0+cu128
Torchattacks 3.5.1
Numpy 2.0.2
Medmnist 3.0.1


##Dataset

data_flag =  
[tissuemnist, pathmnist, chestmnist, dermamnist, octmnist, pnemoniamnist, retinamnist, breastmnist, bloodmnist, tissuemnist, organamnist, organcmnist, organsmnist]

In [6]:
data_flag = 'retinamnist'
# [tissuemnist, pathmnist, chestmnist, dermamnist, octmnist,
# pnemoniamnist, retinamnist, breastmnist, bloodmnist, tissuemnist, organamnist, organcmnist, organsmnist]
download = True

NUM_EPOCHS = 10
BATCH_SIZE = 10
lr = 0.005

info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

print("number of channels : ", n_channels)
print("number of classes : ", n_classes)

number of channels :  3
number of classes :  5


In [7]:
from torchvision.transforms.transforms import Resize
# preprocessing
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Lambda(lambda image: image.convert('RGB')),
    torchvision.transforms.AugMix(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Lambda(lambda image: image.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# load the data
train_dataset = DataClass(split='train', transform=train_transform, download=download)
test_dataset = DataClass(split='test', transform=test_transform, download=download)

# pil_dataset = DataClass(split='train', download=download)

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader_at_eval = data.DataLoader(dataset=train_dataset, batch_size=2*BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

In [8]:
print(train_dataset)
print("===================")
print(test_dataset)

Dataset RetinaMNIST of size 28 (retinamnist)
    Number of datapoints: 1080
    Root location: /root/.medmnist
    Split: train
    Task: ordinal-regression
    Number of channels: 3
    Meaning of labels: {'0': '0', '1': '1', '2': '2', '3': '3', '4': '4'}
    Number of samples: {'train': 1080, 'val': 120, 'test': 400}
    Description: The RetinaMNIST is based on the DeepDRiD challenge, which provides a dataset of 1,600 retina fundus images. The task is ordinal regression for 5-level grading of diabetic retinopathy severity. We split the source training set with a ratio of 9:1 into training and validation set, and use the source validation set as the test set. The source images of 3×1,736×1,824 are center-cropped and resized into 3×28×28.
    License: CC BY 4.0
Dataset RetinaMNIST of size 28 (retinamnist)
    Number of datapoints: 400
    Root location: /root/.medmnist
    Split: test
    Task: ordinal-regression
    Number of channels: 3
    Meaning of labels: {'0': '0', '1': '1', '2'

##Model

MedViTs ---> QMedViT_Softmax_Only

In [9]:
from quantum_variants.softmax_only import QMedViT_Softmax_Only

# Training uses the fast analytic simulator path (qpu_mode=False).
# We'll swap in a QPU-style model with shots after training.
model = QMedViT_Softmax_Only(
    stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
    num_classes=n_classes,
    qpu_mode=False,
).cuda()

initialize_weights...


## Train

In [10]:
# define loss function and optimizer
if task == "multi-label, binary-class":
    criterion = nn.BCEWithLogitsLoss()
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

In [11]:
# train

for epoch in range(NUM_EPOCHS):
    train_correct = 0
    train_total = 0
    test_correct = 0
    test_total = 0
    print('Epoch [%d/%d]'% (epoch+1, NUM_EPOCHS))
    model.train()
    for inputs, targets in tqdm(train_loader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # forward + backward + optimize
        optimizer.zero_grad()
        outputs = model(inputs)

        if task == 'multi-label, binary-class':
            targets = targets.to(torch.float32)
            loss = criterion(outputs, targets)
        else:
            targets = targets.squeeze().long()
            loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

Epoch [1/10]


100%|██████████| 108/108 [00:39<00:00,  2.75it/s]


Epoch [2/10]


100%|██████████| 108/108 [00:39<00:00,  2.76it/s]


Epoch [3/10]


...

100%|██████████| 108/108 [00:38<00:00,  2.78it/s]


Epoch [9/10]


100%|██████████| 108/108 [00:38<00:00,  2.78it/s]


Epoch [10/10]


100%|██████████| 108/108 [00:38<00:00,  2.78it/s]


## Build QPU-style model for evaluation

In [12]:
# qpu_mode=True swaps the U-extraction shortcut for the per-sample
# circuit execution that real hardware requires. Here the device is
# still local (default.qubit) but with finite `qpu_shots`, so we get
# simulated shot-noise — a realistic preview of what running on a
# real QPU would give without paying for hardware time.
model_qpu = QMedViT_Softmax_Only(
    stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
    num_classes=n_classes,
    qpu_mode=True,
    qpu_shots=5000,
    qdevice='default.qubit',
).cuda()

# Copy the parameters trained on the analytic simulator. The forward
# is now stochastic (shot noise) so the same image may give slightly
# different logits across runs.
model_qpu.load_state_dict(model.state_dict())
model_qpu.eval()
print('QPU-sim model ready (shots=5000, device=default.qubit)')

initialize_weights...
QPU-sim model ready (shots=5000, device=default.qubit)


/usr/local/lib/python3.12/dist-packages/pennylane/devices/device_api.py:201: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


##Test

In [22]:
from torch.utils.data import Subset
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import time

SUBSET = 50
import numpy as np
rng = np.random.default_rng(42)
subset_idx = rng.choice(len(test_dataset), size=SUBSET, replace=False).tolist()

subset_loader = data.DataLoader(
    Subset(test_dataset, subset_idx),
    batch_size=2, shuffle=False,
)

split = 'test'
y_score = torch.tensor([])
t0 = time.time()
with torch.no_grad():
    for inputs, targets in tqdm(subset_loader):
        inputs = inputs.cuda()
        outputs = model_qpu(inputs)
        outputs = outputs.softmax(dim=-1)
        y_score = torch.cat((y_score, outputs.cpu()), 0)
elapsed = time.time() - t0
print(f'QPU-sim eval: {len(subset_idx)} samples in {elapsed:.1f}s')

y_true = np.array([test_dataset[i][1] for i in subset_idx]).squeeze()
y_score_np = y_score.numpy()
y_pred = y_score_np.argmax(axis=-1)

acc = accuracy_score(y_true, y_pred)
n_classes_out = y_score_np.shape[-1]
if n_classes_out == 2:
    auc = roc_auc_score(y_true, y_score_np[:, 1])
else:
    auc = roc_auc_score(y_true, y_score_np, multi_class='ovr', average='macro')

print(f'{split}  auc: {auc:.3f}  acc: {acc:.3f}')


100%|██████████| 25/25 [00:23<00:00,  1.08it/s]

QPU-sim eval: 50 samples in 23.2s
test  auc: 0.668  acc: 0.600


In [19]:
# qpu_mode=True swaps the U-extraction shortcut for the per-sample
# circuit execution that real hardware requires. Here the device is
# still local (default.qubit) but with finite `qpu_shots`, so we get
# simulated shot-noise — a realistic preview of what running on a
# real QPU would give without paying for hardware time.
model_qpu_512 = QMedViT_Softmax_Only(
    stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
    num_classes=n_classes,
    qpu_mode=True,
    qpu_shots=512,
    qdevice='default.qubit',
).cuda()

# Copy the parameters trained on the analytic simulator. The forward
# is now stochastic (shot noise) so the same image may give slightly
# different logits across runs.
model_qpu_512.load_state_dict(model.state_dict())
model_qpu_512.eval()
print('QPU-sim model ready (shots=512, device=default.qubit)')

initialize_weights...
QPU-sim model ready (shots=512, device=default.qubit)


In [21]:
split = 'test'

model_qpu_512.eval()
y_true = torch.tensor([])
y_score = torch.tensor([])

data_loader = train_loader_at_eval if split == 'train' else test_loader

with torch.no_grad():
    for inputs, targets in data_loader:
        inputs = inputs.cuda()
        outputs = model_qpu_512(inputs)
        outputs = outputs.softmax(dim=-1)
        y_score = torch.cat((y_score, outputs.cpu()), 0)

    y_score = y_score.detach().numpy()

    evaluator = Evaluator(data_flag, split, size=224)
    metrics = evaluator.evaluate(y_score)

    print('%s  auc: %.3f  acc: %.3f' % (split, *metrics))


test  auc: 0.713  acc: 0.495
